# M2.2.c + M2.2.d: SavGol residual + dayofyear

Implements two features per plan.
Input: train_features.csv
Output: in-memory (not saved to CSV; regenerate in M2.2.e)

- M2.2.c: Residual_savgol_w5p3 = meter_reading - savgol_filter(ffill, w=5, p=3)
- M2.2.d: dayofyear = dt.dayofyear + dt.hour/24

In [ ]:
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter

train = pd.read_csv("../data/raw/train_features.csv")
train = train.sort_values(["building_id", "timestamp"]).reset_index(drop=True)
print(f"Loaded train: {train.shape}")

In [ ]:
results = []
for bid in train["building_id"].unique():
    tmp = train[train["building_id"] == bid].copy()
    # ffill handles mid-series NaN; bfill handles leading NaN;
    # fillna(0) handles all-NaN buildings (rare edge case)
    smoothed = savgol_filter(tmp["meter_reading"].ffill().bfill().fillna(0), 5, 3)
    tmp["Residual_savgol_w5p3"] = (
        tmp["meter_reading"] - smoothed
    )  # original NaN preserved
    results.append(tmp)
train = pd.concat(results).sort_index()
print(f"Train shape now: {train.shape}")
print(f"Residual_savgol_w5p3 column exists: {'Residual_savgol_w5p3' in train.columns}")

In [ ]:
sample_bid = train["building_id"].iloc[0]
sample_residual = train[train["building_id"] == sample_bid]["Residual_savgol_w5p3"]
print(f"Sample building (id={sample_bid}):")
print(f"  Residual mean: {sample_residual.mean():.4f}")
print(f"  Residual std:  {sample_residual.std():.4f}")
print(f"  Residual NaN count: {sample_residual.isna().sum()}")

all_bids = train["building_id"].unique()
bid_means = []
for bid in all_bids:
    m = train[train["building_id"] == bid]["Residual_savgol_w5p3"].mean()
    bid_means.append(m)

print()
print("Across 200 buildings:")
print(f"  Mean of residual means: {np.nanmean(bid_means):.4f}")
print(f"  Max abs residual mean:  {np.nanmax(np.abs(bid_means)):.4f}")
print(
    f"  Buildings with |residual mean| > 1: {sum(abs(m) > 1 for m in bid_means if not np.isnan(m))}"
)

In [ ]:
train["dayofyear"] = (
    pd.to_datetime(train["timestamp"]).dt.dayofyear
    + pd.to_datetime(train["timestamp"]).dt.hour / 24
)

print(f"dayofyear column: dtype={train['dayofyear'].dtype}")
print(f"  Min: {train['dayofyear'].min()}")
print(f"  Max: {train['dayofyear'].max()}")
print(f"  Sample first 5: {train['dayofyear'].head().tolist()}")
print(f"  Sample last 5:  {train['dayofyear'].tail().tolist()}")

In [ ]:
if "anomaly" in train.columns:
    corr = train[["dayofyear", "anomaly"]].corr().iloc[0, 1]
    print(f"dayofyear vs anomaly Pearson corr: {corr:.4f}")
    print("  (paper Fig 5: dayofyear importance=95, ranked #5)")

    bucket = (train["dayofyear"] // 30).astype(int)  # temp; not added to train
    print()
    print("Anomaly rate by ~month bucket:")
    bucket_rates = train.groupby(bucket)["anomaly"].agg(["mean", "count"])
    print(bucket_rates.to_string())

In [ ]:
# Not saving to CSV (same strategy as M2.2.b: regenerate in M2.2.e)
print("Generation complete: Residual_savgol_w5p3 + dayofyear added")
print(f"Total cols now: {train.shape[1]}")
print("  Expected: 59 (raw 57 + Residual_savgol_w5p3 + dayofyear)")
print("Skipping CSV save (regenerate in M2.2.e)")